In [7]:
# CELL 1 — Environment Setup
import time
_t0 = time.time()
try:
    import os
    import sys
    import subprocess

    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "GPU not available. In Colab: Runtime → Change runtime type → Hardware accelerator → GPU (T4)."
        )

    # Install required deps
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "ultralytics==8.3.40", "scikit-image", "-q"]
    )

    # Mount Google Drive (if on Colab)
    try:
        from google.colab import drive  # type: ignore

        if not os.path.exists("/content/drive") or not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        else:
            print("Drive already mounted at /content/drive")
    except Exception as _e:
        print(f"[WARN] Drive mount skipped (not Colab or failed): {_e}")

    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    print("PyTorch:", torch.__version__)
    print("CUDA:", torch.version.cuda)

except Exception as e:
    print(f"[CELL 1 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 1] Time: {time.time() - _t0:.1f}s")


Drive already mounted at /content/drive
GPU: Tesla T4
PyTorch: 2.9.0+cu126
CUDA: 12.6
[CELL 1] Time: 4.8s


In [31]:
# CELL 2 — Link Dataset
import time
_t0 = time.time()
try:
    import os
    from pathlib import Path

    DRIVE_DATASET = "/content/drive/MyDrive/underwater_datasets"
    SPLITS = ["train", "val", "test"]

    # Create working directory /content/turb-detr and cd into it
    WORKDIR = Path("/content/turb-detr")
    WORKDIR.mkdir(parents=True, exist_ok=True)
    os.chdir(str(WORKDIR))
    print("CWD:", os.getcwd())

    # Create data/raw/
    Path("data/raw").mkdir(parents=True, exist_ok=True)

    link_rel = "data/raw/trash-icra19"

    # If symlink already exists, remove it first then recreate
    if os.path.islink(link_rel):
        os.unlink(link_rel)
        print("Removed existing symlink:", link_rel)

    if os.path.exists(link_rel) and (not os.path.islink(link_rel)):
        print(
            f"[CELL 2 ERROR] {link_rel} exists but is not a symlink. Please remove/rename it manually."
        )
    elif not os.path.exists(DRIVE_DATASET):
        mydrive = Path("/content/drive/MyDrive")
        print(f"[CELL 2 ERROR] Drive dataset path not found: {DRIVE_DATASET}")
        if mydrive.exists():
            print("Available paths under /content/drive/MyDrive/:")
            for p in sorted(mydrive.iterdir()):
                print(" -", p)
            print("Please fix DRIVE_DATASET in this cell to the correct folder.")
        else:
            print("/content/drive/MyDrive does not exist. Is Drive mounted?")
    else:
        # If the folder exists but doesn't contain train/val/test, try to auto-detect a nested dataset root
        root = Path(DRIVE_DATASET)
        has_splits = all((root / s).exists() for s in SPLITS)
        if not has_splits:
            candidates = []
            try:
                for child in root.iterdir():
                    if child.is_dir() and all((child / s).exists() for s in SPLITS):
                        candidates.append(child)
            except Exception as _e:
                print(f"[WARN] Could not scan {root}: {_e}")

            if len(candidates) == 1:
                DRIVE_DATASET = str(candidates[0])
                print("[INFO] Auto-detected dataset root:", DRIVE_DATASET)
            else:
                print(
                    "[WARN] DRIVE_DATASET exists but does not contain train/val/test. Contents are:"
                )
                try:
                    for p in sorted(root.iterdir()):
                        print(" -", p)
                except Exception as _e:
                    print(f"[WARN] Could not list contents: {_e}")
                print("Update DRIVE_DATASET to the folder that directly contains train/val/test.")

        # Required symlink (Drive → local)
        os.symlink(DRIVE_DATASET, link_rel)
        print("Symlink created:", link_rel, "->", DRIVE_DATASET)

        link_path = Path(link_rel)
        # Verify by listing train/val/test folders and counting .jpg and .txt
        for split in SPLITS:
            split_dir = link_path / split
            if not split_dir.exists():
                print(f"[WARN] Missing split folder: {split_dir}")
                continue
            jpg_count = len(list(split_dir.glob("*.jpg")))
            txt_count = len(list(split_dir.glob("*.txt")))
            print(f"{split}: {jpg_count} .jpg, {txt_count} .txt")

except Exception as e:
    print(f"[CELL 2 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 2] Time: {time.time() - _t0:.1f}s")


CWD: /content/turb-detr
Removed existing symlink: data/raw/trash-icra19
[WARN] DRIVE_DATASET exists but does not contain train/val/test. Contents are:
 - /content/drive/MyDrive/underwater_datasets/Realworld-Underwater-Image-Enhancement-RUIE-Benchmark-master
 - /content/drive/MyDrive/underwater_datasets/TrashCAN 1.0
 - /content/drive/MyDrive/underwater_datasets/UFO-120
 - /content/drive/MyDrive/underwater_datasets/trash_ICRA19
Update DRIVE_DATASET to the folder that directly contains train/val/test.
Symlink created: data/raw/trash-icra19 -> /content/drive/MyDrive/underwater_datasets
[WARN] Missing split folder: data/raw/trash-icra19/train
[WARN] Missing split folder: data/raw/trash-icra19/val
[WARN] Missing split folder: data/raw/trash-icra19/test
[CELL 2] Time: 0.0s


In [14]:
# CELL 3 — Restructure for Ultralytics
import time
_t0 = time.time()
try:
    import os
    from pathlib import Path

    WORKDIR = Path("/content/turb-detr")
    raw_ds = WORKDIR / "data" / "raw" / "trash-icra19"
    proc_root = WORKDIR / "data" / "processed"

    missing_label_total = 0

    for split in ["train", "val", "test"]:
        src_dir = raw_ds / split
        if not src_dir.exists():
            print(f"[WARN] Source split missing: {src_dir}")
            continue

        out_images = proc_root / split / "images"
        out_labels = proc_root / split / "labels"
        out_images.mkdir(parents=True, exist_ok=True)
        out_labels.mkdir(parents=True, exist_ok=True)

        # Precompute label stems for missing-pair reporting
        label_stems = {p.stem for p in src_dir.glob("*.txt")}

        # Symlink images
        img_count = 0
        for img_path in src_dir.glob("*.jpg"):
            dst = out_images / img_path.name
            if dst.exists() or dst.is_symlink():
                pass
            else:
                try:
                    os.symlink(img_path, dst)
                except FileExistsError:
                    pass
            img_count += 1
            if img_path.stem not in label_stems:
                missing_label_total += 1

        # Symlink labels (skip .xml entirely)
        lbl_count = 0
        for lbl_path in src_dir.glob("*.txt"):
            dst = out_labels / lbl_path.name
            if dst.exists() or dst.is_symlink():
                pass
            else:
                try:
                    os.symlink(lbl_path, dst)
                except FileExistsError:
                    pass
            lbl_count += 1

        print(f"{split}: {img_count} images, {lbl_count} labels")

    if missing_label_total > 0:
        print(f"[WARN] Images missing matching .txt labels: {missing_label_total}")

except Exception as e:
    print(f"[CELL 3 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 3] Time: {time.time() - _t0:.1f}s")


[WARN] Source split missing: /content/turb-detr/data/raw/trash-icra19/train
[WARN] Source split missing: /content/turb-detr/data/raw/trash-icra19/val
[WARN] Source split missing: /content/turb-detr/data/raw/trash-icra19/test
[CELL 3] Time: 0.0s


In [16]:
# CELL 4 — Class Distribution
import time
_t0 = time.time()
try:
    from pathlib import Path

    WORKDIR = Path("/content/turb-detr")
    labels_dir = WORKDIR / "data" / "processed" / "train" / "labels"

    class_counts = {}
    bad_files = 0
    bad_lines = 0
    max_class_id = -1
    total_boxes = 0

    label_files = list(labels_dir.glob("*.txt"))
    print("Train label files:", len(label_files))

    for lf in label_files:
        try:
            text = lf.read_text().strip()
            if not text:
                continue
            for line in text.splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    bad_lines += 1
                    continue
                try:
                    cid = int(float(parts[0]))
                except Exception:
                    bad_lines += 1
                    continue
                class_counts[cid] = class_counts.get(cid, 0) + 1
                max_class_id = max(max_class_id, cid)
                total_boxes += 1
        except Exception:
            bad_files += 1

    for cid in sorted(class_counts.keys()):
        print(f"Class {cid}: {class_counts[cid]} boxes")

    total_classes = len(class_counts)
    print("Total classes:", total_classes)
    print("Total boxes:", total_boxes)
    print("Bad label files skipped:", bad_files)
    print("Bad label lines skipped:", bad_lines)

    # Persist for next cells
    CLASS_COUNTS = class_counts
    MAX_CLASS_ID = max_class_id
    TOTAL_CLASSES = total_classes
    TOTAL_BOXES = total_boxes

except Exception as e:
    print(f"[CELL 4 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 4] Time: {time.time() - _t0:.1f}s")


Train label files: 0
Total classes: 0
Total boxes: 0
Bad label files skipped: 0
Bad label lines skipped: 0
[CELL 4] Time: 0.0s


In [17]:
# CELL 5 — Create Dataset YAML
import time
_t0 = time.time()
try:
    from pathlib import Path

    WORKDIR = Path("/content/turb-detr")
    proc_root = WORKDIR / "data" / "processed"

    # Determine class names based on Cell 4 output
    try:
        max_id = int(MAX_CLASS_ID)  # from Cell 4
    except Exception:
        max_id = 0

    if max_id <= 0:
        CLASS_NAMES = {0: "debris"}
    else:
        CLASS_NAMES = {i: f"class_{i}" for i in range(max_id + 1)}

    cfg_dir = WORKDIR / "configs" / "datasets"
    cfg_dir.mkdir(parents=True, exist_ok=True)

    yaml_path = cfg_dir / "trash_icra19.yaml"

    data_dict = {
        "path": str(proc_root.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "names": CLASS_NAMES,
    }

    # Write YAML
    try:
        import yaml  # PyYAML (usually installed via Ultralytics)

        yaml_path.write_text(yaml.safe_dump(data_dict, sort_keys=False))
    except Exception as _e:
        # Fallback: minimal YAML writer (to avoid hard failure)
        print(f"[WARN] yaml module not available or failed ({_e}); writing YAML manually")
        lines = []
        lines.append(f"path: {data_dict['path']}")
        lines.append(f"train: {data_dict['train']}")
        lines.append(f"val: {data_dict['val']}")
        lines.append(f"test: {data_dict['test']}")
        lines.append("names:")
        for k in sorted(CLASS_NAMES.keys()):
            lines.append(f"  {k}: {CLASS_NAMES[k]}")
        yaml_path.write_text("\n".join(lines) + "\n")

    print("Wrote:", yaml_path)
    print("--- YAML ---")
    print(yaml_path.read_text())

except Exception as e:
    print(f"[CELL 5 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 5] Time: {time.time() - _t0:.1f}s")


Wrote: /content/turb-detr/configs/datasets/trash_icra19.yaml
--- YAML ---
path: /content/turb-detr/data/processed
train: train/images
val: val/images
test: test/images
names:
  0: debris

[CELL 5] Time: 0.0s


In [19]:
# CELL 6 — Set Seeds
import time
_t0 = time.time()
try:
    import os
    import random

    import numpy as np
    import torch

    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    os.environ["PYTHONHASHSEED"] = "42"
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print("Seeds set to 42")

except Exception as e:
    print(f"[CELL 6 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 6] Time: {time.time() - _t0:.1f}s")


Seeds set to 42
[CELL 6] Time: 0.0s


In [21]:
# CELL 7 — Train Baseline
import time
_t0 = time.time()
try:
    from pathlib import Path

    from ultralytics import RTDETR

    WORKDIR = Path("/content/turb-detr")
    yaml_path = WORKDIR / "configs" / "datasets" / "trash_icra19.yaml"

    if not yaml_path.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {yaml_path}")

    # Quick sanity check: ensure dataset is actually present
    train_images_dir = WORKDIR / "data" / "processed" / "train" / "images"
    if not train_images_dir.exists() or len(list(train_images_dir.glob("*.jpg"))) == 0:
        raise RuntimeError(
            "No training images found in data/processed/train/images. "
            "Fix DRIVE_DATASET in Cell 2 and rerun Cells 2–3."
        )

    # Ensure RT-DETR-S weights are available (Ultralytics should download, but handle offline cases)
    weights_name = "rtdetr-s.pt"  # MUST be rtdetr-s.pt
    if not Path(weights_name).exists():
        try:
            from ultralytics.utils.downloads import attempt_download_asset

            print(f"[INFO] '{weights_name}' not found locally. Attempting download...")
            attempt_download_asset(weights_name)
        except Exception as _e:
            print(f"[WARN] Could not auto-download {weights_name}: {type(_e).__name__}: {_e}")

    if not Path(weights_name).exists():
        raise FileNotFoundError(
            f"{weights_name} not found. Ensure Colab has internet access, then rerun Cell 7."
        )

    model = RTDETR(weights_name)

    train_t0 = time.time()
    try:
        model.train(
            data=str(yaml_path),
            epochs=30,
            imgsz=640,
            batch=16,
            seed=42,
            project="results",
            name="baseline_rtdetr_clean",
            device=0,
            workers=2,
            patience=10,
            verbose=True,
        )
    except RuntimeError as _e:
        msg = str(_e)
        if "out of memory" in msg.lower():
            print(
                "[CUDA OOM] Training ran out of GPU memory. Reduce batch to 8 and rerun CELL 7."
            )
            raise
        else:
            raise
    finally:
        print(f"Training call time: {time.time() - train_t0:.1f}s")

    best_pt = WORKDIR / "results" / "baseline_rtdetr_clean" / "weights" / "best.pt"
    if best_pt.exists():
        print("Found best weights:", best_pt)
    else:
        print("[WARN] best.pt not found at:", best_pt)
        last_pt = WORKDIR / "results" / "baseline_rtdetr_clean" / "weights" / "last.pt"
        print("last.pt exists:", last_pt.exists())

except Exception as e:
    print(f"[CELL 7 ERROR] {type(e).__name__}: {e}")
    # If this looks like a dataset error, print YAML contents
    try:
        from pathlib import Path

        _yp = Path("/content/turb-detr/configs/datasets/trash_icra19.yaml")
        if _yp.exists():
            print("--- Dataset YAML ---")
            print(_yp.read_text())
    except Exception as _e2:
        print(f"[CELL 7] Could not print YAML: {_e2}")
finally:
    print(f"[CELL 7] Time: {time.time() - _t0:.1f}s")


[CELL 7 ERROR] FileNotFoundError: [Errno 2] No such file or directory: 'rtdetr-s.pt'
--- Dataset YAML ---
path: /content/turb-detr/data/processed
train: train/images
val: val/images
test: test/images
names:
  0: debris

[CELL 7] Time: 0.3s


In [23]:
# CELL 8 — Evaluate on Clean Test Set
import time
_t0 = time.time()
try:
    from pathlib import Path

    from ultralytics import RTDETR

    WORKDIR = Path("/content/turb-detr")
    yaml_path = WORKDIR / "configs" / "datasets" / "trash_icra19.yaml"

    best_pt = WORKDIR / "results" / "baseline_rtdetr_clean" / "weights" / "best.pt"
    last_pt = WORKDIR / "results" / "baseline_rtdetr_clean" / "weights" / "last.pt"

    weights_path = None
    if best_pt.exists():
        weights_path = best_pt
    elif last_pt.exists():
        weights_path = last_pt
    else:
        print("[CELL 8 ERROR] Neither best.pt nor last.pt exists. Run CELL 7 first.")

    if weights_path is not None:
        model = RTDETR(str(weights_path))
        results = model.val(data=str(yaml_path), split="test", device=0)

        clean_map50 = float(results.box.map50)
        clean_map50_95 = float(results.box.map)
        clean_prec = float(results.box.mp)
        clean_rec = float(results.box.mr)

        print(f"Clean mAP@0.5:      {clean_map50:.4f}")
        print(f"Clean mAP@0.5:0.95: {clean_map50_95:.4f}")
        print(f"Clean Precision:    {clean_prec:.4f}")
        print(f"Clean Recall:       {clean_rec:.4f}")

        # Persist for later cells
        CLEAN_METRICS = {
            "map50": clean_map50,
            "map50_95": clean_map50_95,
            "precision": clean_prec,
            "recall": clean_rec,
        }

except Exception as e:
    print(f"[CELL 8 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 8] Time: {time.time() - _t0:.1f}s")


[CELL 8 ERROR] Neither best.pt nor last.pt exists. Run CELL 7 first.
[CELL 8] Time: 0.0s


In [27]:
# CELL 9 — Generate Turbid Test Images
import time
_t0 = time.time()
try:
    import os
    from pathlib import Path

    import cv2
    import numpy as np
    from tqdm import tqdm

    # DO NOT MODIFY (as requested)
    def jaffe_mcglamery_turbidity(image, c=0.15, depth=3.0, seed=None):
        import numpy as np
        rng = np.random.RandomState(seed)
        img = image.astype(np.float32) / 255.0
        attenuation = np.exp(-c * depth)
        backscatter = np.zeros_like(img)
        backscatter[:, :, 0] = 0.10 * (1 - attenuation)
        backscatter[:, :, 1] = 0.30 * (1 - attenuation)
        backscatter[:, :, 2] = 0.05 * (1 - attenuation)
        turbid = img * attenuation + backscatter
        noise = rng.normal(0, 0.02 * c * 10, img.shape).astype(np.float32)
        return (np.clip(turbid + noise, 0, 1) * 255).astype(np.uint8)

    WORKDIR = Path("/content/turb-detr")
    src_images = WORKDIR / "data" / "processed" / "test" / "images"
    src_labels = WORKDIR / "data" / "processed" / "test" / "labels"

    out_root = WORKDIR / "data" / "augmented"

    levels = {
        "light": 0.05,
        "medium": 0.15,
        "heavy": 0.30,
    }

    test_imgs = sorted(src_images.glob("*.jpg"))
    print("Test images:", len(test_imgs))

    total_imread_fail = 0

    for level, c_val in levels.items():
        out_images = out_root / level / "images"
        out_labels = out_root / level / "labels"
        out_images.mkdir(parents=True, exist_ok=True)
        out_labels.mkdir(parents=True, exist_ok=True)

        level_imread_fail = 0
        level_written = 0
        level_missing_labels = 0

        for i, img_path in enumerate(tqdm(test_imgs, desc=f"turbid-{level}")):
            img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
            if img is None:
                level_imread_fail += 1
                continue

            turbid = jaffe_mcglamery_turbidity(img, c=c_val, depth=3.0, seed=42 + i)
            out_img_path = out_images / img_path.name
            ok = cv2.imwrite(str(out_img_path), turbid)
            if ok:
                level_written += 1

            # Link matching label
            lbl_path = src_labels / (img_path.stem + ".txt")
            out_lbl_path = out_labels / (img_path.stem + ".txt")
            if lbl_path.exists():
                if not (out_lbl_path.exists() or out_lbl_path.is_symlink()):
                    try:
                        os.symlink(lbl_path, out_lbl_path)
                    except FileExistsError:
                        pass
            else:
                level_missing_labels += 1

        total_imread_fail += level_imread_fail
        print(
            f"{level}: wrote {level_written} images, imread_fail={level_imread_fail}, missing_labels={level_missing_labels}"
        )

    if total_imread_fail > 0:
        print("Total cv2.imread failures:", total_imread_fail)

except Exception as e:
    print(f"[CELL 9 ERROR] {type(e).__name__}: {e}")
    print("If this is 'No module named cv2', install opencv-python-headless and rerun CELL 9.")
finally:
    print(f"[CELL 9] Time: {time.time() - _t0:.1f}s")


Test images: 0


turbid-light: 0it [00:00, ?it/s]


light: wrote 0 images, imread_fail=0, missing_labels=0


turbid-medium: 0it [00:00, ?it/s]


medium: wrote 0 images, imread_fail=0, missing_labels=0


turbid-heavy: 0it [00:00, ?it/s]

heavy: wrote 0 images, imread_fail=0, missing_labels=0
[CELL 9] Time: 0.0s


In [28]:
# CELL 10 — Visualize Turbidity (Paper Figure)
import time
_t0 = time.time()
try:
    from pathlib import Path

    import cv2
    import matplotlib.pyplot as plt

    WORKDIR = Path("/content/turb-detr")
    clean_dir = WORKDIR / "data" / "processed" / "test" / "images"

    test_imgs = sorted(clean_dir.glob("*.jpg"))
    if not test_imgs:
        raise FileNotFoundError(f"No test images found in {clean_dir}")

    img_path = test_imgs[0]
    name = img_path.name

    def _read_rgb(p: Path):
        bgr = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if bgr is None:
            return None
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    clean = _read_rgb(img_path)
    light = _read_rgb(WORKDIR / "data" / "augmented" / "light" / "images" / name)
    medium = _read_rgb(WORKDIR / "data" / "augmented" / "medium" / "images" / name)
    heavy = _read_rgb(WORKDIR / "data" / "augmented" / "heavy" / "images" / name)

    if clean is None:
        raise RuntimeError(f"Failed to read clean image: {img_path}")

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(clean)
    axes[0].set_title("Clean")
    axes[1].imshow(light if light is not None else clean)
    axes[1].set_title("Light (c=0.05)")
    axes[2].imshow(medium if medium is not None else clean)
    axes[2].set_title("Medium (c=0.15)")
    axes[3].imshow(heavy if heavy is not None else clean)
    axes[3].set_title("Heavy (c=0.30)")

    for ax in axes:
        ax.axis("off")

    out_path = WORKDIR / "results" / "figures" / "turbidity_levels.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(out_path), dpi=150, bbox_inches="tight")
    print("Saved:", out_path)
    plt.show()

except Exception as e:
    print(f"[CELL 10 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 10] Time: {time.time() - _t0:.1f}s")


[CELL 10 ERROR] FileNotFoundError: No test images found in /content/turb-detr/data/processed/test/images
[CELL 10] Time: 0.0s


In [29]:
# CELL 11 — Evaluate Baseline on Turbid Test Sets
import time
_t0 = time.time()
try:
    from pathlib import Path

    from ultralytics import RTDETR

    WORKDIR = Path("/content/turb-detr")
    cfg_dir = WORKDIR / "configs" / "datasets"
    cfg_dir.mkdir(parents=True, exist_ok=True)

    # Ensure CLASS_NAMES exists (from Cell 5)
    try:
        _ = CLASS_NAMES
    except Exception:
        CLASS_NAMES = {0: "debris"}
        print("[WARN] CLASS_NAMES not found; defaulting to {0: 'debris'}. Run CELL 5 for correct names.")

    best_pt = WORKDIR / "results" / "baseline_rtdetr_clean" / "weights" / "best.pt"
    last_pt = WORKDIR / "results" / "baseline_rtdetr_clean" / "weights" / "last.pt"
    weights_path = best_pt if best_pt.exists() else (last_pt if last_pt.exists() else None)
    if weights_path is None:
        raise FileNotFoundError("No baseline weights found (best.pt/last.pt). Run CELL 7.")

    model = RTDETR(str(weights_path))

    turbid_metrics = {}

    for level in ["light", "medium", "heavy"]:
        try:
            aug_path = (WORKDIR / "data" / "augmented" / level).resolve()
            yaml_path = cfg_dir / f"trash_icra19_turbid_{level}.yaml"

            data_dict = {
                "path": str(aug_path),
                "train": "images",
                "val": "images",
                "test": "images",
                "names": CLASS_NAMES,
            }

            try:
                import yaml

                yaml_path.write_text(yaml.safe_dump(data_dict, sort_keys=False))
            except Exception as _e:
                print(f"[WARN] yaml write failed ({_e}); writing manually for {level}")
                lines = []
                lines.append(f"path: {data_dict['path']}")
                lines.append("train: images")
                lines.append("val: images")
                lines.append("test: images")
                lines.append("names:")
                for k in sorted(CLASS_NAMES.keys()):
                    lines.append(f"  {k}: {CLASS_NAMES[k]}")
                yaml_path.write_text("\n".join(lines) + "\n")

            results = model.val(data=str(yaml_path), split="test", device=0)
            m = {
                "map50": float(results.box.map50),
                "map50_95": float(results.box.map),
                "precision": float(results.box.mp),
                "recall": float(results.box.mr),
            }
            turbid_metrics[level] = m
            print(f"{level} mAP@0.5: {m['map50']:.4f}")

        except Exception as _e:
            print(f"[CELL 11 ERROR] level={level}: {type(_e).__name__}: {_e}")
            continue

    TURBID_METRICS = turbid_metrics

except Exception as e:
    print(f"[CELL 11 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 11] Time: {time.time() - _t0:.1f}s")


[CELL 11 ERROR] FileNotFoundError: No baseline weights found (best.pt/last.pt). Run CELL 7.
[CELL 11] Time: 0.0s


In [30]:
# CELL 12 — Final Results Table
import time
_t0 = time.time()
try:
    def _fmt(x):
        try:
            return f"{float(x):.4f}"
        except Exception:
            return "NA"

    clean = globals().get("CLEAN_METRICS", None)
    turbid = globals().get("TURBID_METRICS", {})

    if clean is None:
        print("[CELL 12 ERROR] CLEAN_METRICS not found. Run CELL 8.")
        clean = {"map50": float('nan'), "map50_95": float('nan'), "precision": float('nan'), "recall": float('nan')}

    clean_map50 = clean.get("map50", float('nan'))

    rows = []
    rows.append(("Clean", clean))
    rows.append(("Turbid-light", turbid.get("light", {})))
    rows.append(("Turbid-medium", turbid.get("medium", {})))
    rows.append(("Turbid-heavy", turbid.get("heavy", {})))

    print("Condition\tmAP@0.5\tmAP@0.5:0.95\tPrecision\tRecall\tDrop%")
    for name, m in rows:
        map50 = m.get("map50", float('nan'))
        drop = None
        try:
            drop = (clean_map50 - map50) / clean_map50 * 100.0
        except Exception:
            drop = float('nan')

        print(
            f"{name}\t{_fmt(map50)}\t{_fmt(m.get('map50_95', float('nan')))}\t{_fmt(m.get('precision', float('nan')))}\t{_fmt(m.get('recall', float('nan')))}\t{_fmt(drop)}"
        )

    heavy_map50 = turbid.get("heavy", {}).get("map50", None)
    heavy_drop = None
    try:
        heavy_drop = (clean_map50 - heavy_map50) / clean_map50 * 100.0
    except Exception:
        heavy_drop = None

    if heavy_drop is None:
        print("Verdict: Could not compute heavy drop (missing metrics).")
    else:
        if heavy_drop > 15:
            print("Verdict: SIGNIFICANT DROP. Research problem VALIDATED.")
        elif 10 <= heavy_drop <= 15:
            print("Verdict: MODERATE DROP. Publishable — emphasize per-class analysis.")
        else:
            print("Verdict: Drop < 10%. Consider stronger turbidity or harder conditions.")

except Exception as e:
    print(f"[CELL 12 ERROR] {type(e).__name__}: {e}")
finally:
    print(f"[CELL 12] Time: {time.time() - _t0:.1f}s")


[CELL 12 ERROR] CLEAN_METRICS not found. Run CELL 8.
Condition	mAP@0.5	mAP@0.5:0.95	Precision	Recall	Drop%
Clean	nan	nan	nan	nan	nan
Turbid-light	nan	nan	nan	nan	nan
Turbid-medium	nan	nan	nan	nan	nan
Turbid-heavy	nan	nan	nan	nan	nan
Verdict: Could not compute heavy drop (missing metrics).
[CELL 12] Time: 0.0s
